# 03. Multiscale packing: пилот с замороженными признаками

## 1. Научный вопрос и три модели
Помогает ли явное представление локальной упаковки предсказывать ΔΔG малым MLP?
У всех моделей одна архитектура: **388 → 32 → 256 → 21**, Tanh между линейными слоями,
**26 293** обучаемых параметра, одинаковые начальные веса и порядок белков при одном seed.

| Модель | Дополнение к исходным 384 признакам |
|---|---|
| `baseline_mlp` | четыре точных нуля |
| `density_mlp` | нормированные c6, c12, c18 и точный ноль |
| `fractal_mlp` | нормированные c6, c12, c18, d |

Основное сравнение: **D_fractal = macro MAE(density) − macro MAE(fractal)**;
положительное значение в пользу явного показателя масштабирования.
Вторичные сравнения: baseline − density и baseline − fractal.
Предсказание одиночной замены: `score[position, mutant] − score[position, WT]`;
алфавит `ACDEFGHIKLMNPQRSTVWYX`, целевая величина **−ddG_ML**, ккал/моль,
положительный знак означает дестабилизацию. Сохраняются фильтры одиночных стандартных замен,
проверки WT/PDB и полной мутантной последовательности исходного пилота.
ThermoMPNN не дообучается; используются только готовые 384-мерные признаки.

Это исследование представления признаков, а не проверка ускорения нейросетей
фрактальной геометрией или доказательство фрактального физического механизма.

## 2. Настройки пользователя
По умолчанию **SMOKE / NON-SCIENTIFIC**: четыре заранее выбранных train-белка,
два validation-белка, seed 42, две эпохи, все три модели. Это только техническая проверка.
Полный пилот: `RUN_MODE = "full"`, новый `RUN_ID`, `CONFIRM_FULL_RUN = True`;
209 train / 31 validation, seeds 42–44, 20 эпох.
Каждый независимый запуск требует нового RUN_ID. `RESUME = True` пропускает целиком
завершённые совместимые пары seed/model; прерванная пара обучается заново.
Возобновление требует неизменных исходников, данных и настроек.

In [ ]:
RUN_MODE = "smoke"
RUN_ID = "fractal_pilot_smoke_01"
DEVICE = "cpu"
RESUME = False
CONFIRM_FULL_RUN = False

## 3. Среда и существующий кеш
Запускайте из корня репозитория или `notebooks` в среде `NeuroPP` (`environment.yml`).
Нужны NumPy, pandas, PyTorch и matplotlib. Модуль не импортирует графовый проект.
Кеш читается без записи и без пересчёта признаков. Проверяются имя белка, WT,
порядок аминокислот, размер 384, полные координаты, SHA256 PDB и подпись признаков.
Старые baseline scores и предсказания не используются; mmap не читает их хранилища.
Отсутствующие кеш/PDB/CSV вызывают ошибку: скачивание и извлечение архивов отключены.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))
import fractal_pilot as fp

settings = fp.experiment_settings(RUN_MODE, CONFIRM_FULL_RUN)
fp.require(DEVICE == "cpu", "Используйте DEVICE = 'cpu'")
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.use_deterministic_algorithms(True)
CACHE_DIR = ROOT / "data/features/thermompnn"
PDB_DIR = ROOT / "data/megascale/AlphaFold_model_PDBs"
CSV_PATH = ROOT / "data/megascale/Processed_K50_dG_datasets/Tsuboyama2023_Dataset2_Dataset3_20230416.csv"
for path in (CACHE_DIR, PDB_DIR, CSV_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Нет существующего локального артефакта: {path}")
print("FULL: exploratory pilot" if settings["scientific"] else "SMOKE — NON-SCIENTIFIC")
print("PyTorch:", torch.__version__, "feature signature:", fp.FEATURE_SIGNATURE)

## 4. Зафиксированное разбиение и загрузка development
Файл `splits/fractal_pilot.json` — точная побайтовая копия locked split из `swarm-advanced`.
Проверяем SHA256 `310839d388e4430a817f849fb9c3f08ca563bc074706bb9b9facffbcb1aeb796`.
Численности: 209 train, 31 validation, 30 head_holdout, 28 legacy_test;
1A32 остаётся в train, 2MA4 — в head_holdout.
Загрузчик допускает только train/validation: сначала читает столбец идентификаторов,
затем пропускает все ненужные строки **до разбора меток**. Защищённые метки не загружаются.
CSV с многострочными записями отклоняется до второго прохода.

In [ ]:
split = fp.load_locked_split(ROOT / "splits/fractal_pilot.json")
train_names, val_names = fp.selected_names(split, settings)
print("Проверенный SHA256:", fp.SPLIT_SHA256)
print("Используемые белки:", len(train_names), "train /", len(val_names), "validation")

## 5. Многомасштабные дескрипторы упаковки
Для WT Cα-координат считаем евклидовы расстояния до всех остатков того же белка.
`M_i(r) = 1 + число других остатков с расстоянием <= r`, радиусы **6, 12, 18 Å**.
`c6 = log M(6)`, `c12 = log M(12)`, `c18 = log M(18)` (натуральный логарифм),
`d = (c12 − c6) / log(2)`.

**d — конечномасштабный локальный показатель упаковки, не установленная фрактальная размерность.**
Он алгебраически определяется c6 и c12 и не содержит независимой структурной информации:
это явно заданное производное представление нелинейных логарифмических счётчиков;
в самих логарифмических каналах d является линейной комбинацией.

Геометрия вычисляется один раз на WT-белок и переиспользуется для всех мутаций.
Даже в smoke нормировка использует остатки **всех 209 train-белков** (без их меток);
поэтому нужны их локальные WT PDB. Validation не участвует в подгонке нормировки.

In [ ]:
geometry_names = list(split["train"]) + val_names
structures = fp.load_geometry(geometry_names, split, PDB_DIR)
normalization = fp.fit_normalization(structures, split)
records = fp.load_records(train_names + val_names, split, structures, CACHE_DIR, CSV_PATH)
print("Загружены только выбранные development-мутации; геометрия train готова.")

## 6. Распределения и нормировка
Среднее и population std (ddof=0) вычисляются по остаткам train, каждый остаток имеет равный вес.
Нулевая std заменяется масштабом 1. Замороженные 384 признака сохраняются без изменений.
Ниже — только геометрия и агрегаты, индивидуальные метки не выводятся.
Параметры нормировки сохраняются в manifest.

In [ ]:
normalization_table = pd.DataFrame({key: normalization[key] for key in ("mean", "std", "scale")},
                                   index=fp.CHANNELS)
display(normalization_table)
print("Белков / остатков для нормировки:", normalization["proteins"], normalization["residues"])
fig, axes = plt.subplots(1, 4, figsize=(12, 2.6))
for names, label in ((split["train"], "train (209)"), (val_names, "validation")):
    values = np.concatenate([structures[name]["geometry"] for name in names])
    normalized = (values - normalization_table["mean"].to_numpy()) / normalization_table["scale"].to_numpy()
    for i, ax in enumerate(axes):
        ax.hist(normalized[:, i], bins=25, density=True, histtype="step", label=label)
        ax.set(title=fp.CHANNELS[i], xlabel="Нормированное значение")
axes[0].legend()
fig.tight_layout()
plt.show()

## 7. Равенство моделей
Одинаковый набор зарегистрированных параметров; выключенные каналы остаются строго нулевыми.
При одном seed все веса трёх вариантов изначально побитово совпадают.

In [ ]:
heads = {variant: fp.build_head(variant, 42) for variant in fp.VARIANTS}
reference_state = heads["baseline_mlp"].state_dict()
for head in heads.values():
    fp.require(all(torch.equal(value, head.state_dict()[key]) for key, value in reference_state.items()),
               "Начальные веса различаются")
display(pd.DataFrame([{"model": variant, "trainable_parameters": sum(p.numel() for p in head.parameters())}
                      for variant, head in heads.items()]))
del heads, reference_state

## 8. ОБУЧЕНИЕ — запускать осознанно
Следующая ячейка запускает обучение, включая smoke на реальных development-данных.
При подготовке этого notebook она **не выполнялась**.
Adam lr=0.001; mean MSE по всем валидным мутациям одного белка; один шаг на белок за эпоху.
Нет dropout, scheduler или дообучения backbone. Для одного seed порядок белков и начальные веса
совпадают между вариантами. Checkpoint выбирается по validation protein macro MAE,
точное равенство оставляет более раннюю эпоху. Pooled MAE/RMSE — вторичные метрики.

Создаются только manifest и четыре сводных CSV в `results/fractal_pilot/<RUN_ID>/`,
лучшие веса — в `weights/fractal_pilot/<RUN_ID>/`. Manifest содержит ревизию/хеш исходников,
идентичность входов, настройки геометрии, нормировку и выбранные эпохи.
Существующий каталог защищён; для совместимого продолжения задайте RESUME.

In [ ]:
run = fp.run_experiment(ROOT, RUN_ID, settings, split, train_names, val_names,
                        structures, records, normalization, CSV_PATH,
                        resume=RESUME, device=DEVICE)

## 9. Кривые validation
Показываем protein macro MAE по эпохам для всех seed; эпохи сравнимы между вариантами.
Эти данные используются для выбора checkpoint и не являются независимой итоговой оценкой.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, variant in zip(axes, fp.VARIANTS):
    for seed, history in run["history"].query("model == @variant").groupby("seed"):
        ax.plot(history["epoch"], history["macro_MAE"], marker=".", label=f"seed {seed}")
    ax.set(title=variant, xlabel="Эпоха")
axes[0].set_ylabel("Validation macro MAE, ккал/моль")
axes[-1].legend()
fig.tight_layout()
plt.show()

## 10. Сравнение macro MAE
Macro MAE — среднее MAE по белкам с одинаковым весом каждого белка.
Pooled метрики взвешивают мутации. Не выбираем лучший seed и не строим ансамбль.

In [ ]:
display(run["summary"].round(6))
model_means = run["summary"].groupby("model").agg(
    macro_MAE_mean=("macro_MAE", "mean"), macro_MAE_std=("macro_MAE", "std"),
    pooled_MAE_mean=("pooled_MAE", "mean"), pooled_RMSE_mean=("pooled_RMSE", "mean")
).reindex(fp.VARIANTS)
display(model_means.round(6))

## 11. Парные эффекты по seed
`D_fractal` — основной эффект density − fractal; `D_density` — baseline − density;
`D_total` — baseline − fractal. Положительный эффект означает меньшую ошибку второй модели.
В smoke один seed: std не определена, научные выводы недопустимы.

In [ ]:
effects = fp.paired_effects(run["summary"])
display(effects.round(6))

## 12. По-белковое сравнение и интерпретация
Сначала усредняем MAE каждого белка по seed. Показываем агрегаты, а не индивидуальные метки.
Согласованный положительный D_fractal поддержал бы полезность явного показателя в данном MLP.
Различия между белками показывают неоднородность эффекта. Улучшение относительно baseline
само по себе не выделяет пользу d: основное сравнение — density против fractal.
Три seed недостаточны для определённого статистического вывода; validation использована
для выбора эпохи. Это исследовательский пилот. Оценки head_holdout/legacy_test здесь нет.

In [ ]:
protein_panel = run["per_protein"].groupby(["WT_name", "model"])["MAE"].mean().unstack("model")
protein_panel["D_fractal"] = protein_panel["density_mlp"] - protein_panel["fractal_mlp"]
display(protein_panel.round(6))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(range(3), model_means["macro_MAE_mean"], alpha=0.5)
for i, variant in enumerate(fp.VARIANTS):
    scores = run["summary"].query("model == @variant")["macro_MAE"]
    axes[0].scatter(np.full(len(scores), i), scores, color="black", s=15)
axes[0].set(xticks=range(3), xticklabels=["baseline", "density", "fractal"],
            ylabel="Validation macro MAE, ккал/моль", title="Среднее и отдельные seed")
x, y = protein_panel["density_mlp"], protein_panel["fractal_mlp"]
axes[1].scatter(x, y, s=20)
low, high = min(x.min(), y.min()), max(x.max(), y.max())
axes[1].plot([low, high], [low, high], color="gray", linestyle="--")
axes[1].set(xlabel="MAE density, ккал/моль", ylabel="MAE fractal, ккал/моль",
            title="Белки: ниже диагонали — польза d")
fig.suptitle("SMOKE — NON-SCIENTIFIC" if RUN_MODE == "smoke" else "Exploratory validation pilot")
fig.tight_layout()
plt.show()